# 01 — EDA & data-quality decisions
Narrative around `sql/01_raw.sql` + `sql/02_staging.sql`. Decisions made here are
promoted into the SQL; this notebook is the record of *why*.

Prereq: `python run_pipeline.py raw stage` has been run.

In [1]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.config import load, sql_params
from src.db import connect

cfg = load()
params = sql_params(cfg)
con = connect(cfg["paths"]["duckdb"], read_only=True)

## Raw shape

In [2]:
raw_n = con.execute("SELECT count(*) FROM raw_loans").fetchone()[0]
print(f"raw_loans: {raw_n:,} rows")

raw_loans: 2,260,701 rows


## Maturity pre-flight (36-month loans) — how the window was chosen
Newest `issue_month` with `frac_terminal >= 0.98` sets the end of the modelling window.

In [3]:
sql = (ROOT / "sql" / "preflight_maturity.sql").read_text().replace("${term_months}", params["term_months"])
preflight = con.execute(sql).fetchdf()
preflight.tail(24)

,issue_month,n,frac_terminal,frac_charged_off
115,2017-01,24152,0.5349,0.1081
116,2017-02,20453,0.5092,0.1041
117,2017-03,27805,0.4762,0.0939
118,2017-04,21864,0.4605,0.0966
119,2017-05,27456,0.4516,0.0961
120,2017-06,27785,0.4194,0.0862
121,2017-07,28738,0.3954,0.0822
122,2017-08,32595,0.3709,0.0746
123,2017-09,26894,0.3537,0.0733
124,2017-10,26889,0.3285,0.0613


In [4]:
thr = cfg["maturity"]["terminal_fraction_threshold"]
mature = preflight[preflight.frac_terminal >= thr]
print(f"newest month with frac_terminal >= {thr}: {mature.issue_month.iloc[-1]}")
print(f"config: oot_cutoff={cfg['split']['oot_cutoff']}  issue_year_max={cfg['maturity']['issue_year_max']}"
      f"  min_months_since_issue={cfg['maturity']['min_months_since_issue']}")

newest month with frac_terminal >= 0.98: 2016-02
config: oot_cutoff=2015-01-01  issue_year_max=2016  min_months_since_issue=34


## Staged table — column profile

In [5]:
stg = con.execute("SELECT * FROM stg_loans").fetchdf()
print(f"stg_loans: {len(stg):,} rows  ({len(stg)/raw_n:.1%} of raw)")
pd.DataFrame({
    "dtype": stg.dtypes.astype(str),
    "null_frac": stg.isna().mean().round(4),
    "n_unique": stg.nunique(),
}).sort_values("null_frac", ascending=False)

stg_loans: 640,919 rows  (28.4% of raw)


,dtype,null_frac,n_unique
mths_since_recent_inq,float64,0.1161,26
emp_length_years,Int32,0.0610,11
pct_tl_nvr_dlq,float64,0.0354,560
mo_sin_old_rev_tl_op,float64,0.0352,726
tot_coll_amt,float64,0.0352,9402
num_tl_op_past_12m,float64,0.0352,30
num_actv_bc_tl,float64,0.0352,31
tot_hi_cred_lim,float64,0.0352,286118
num_tl_90g_dpd_24m,float64,0.0352,24
avg_cur_bal,float64,0.0352,62003


## Categorical sanity — standardised values only?

In [6]:
for c in ["home_ownership", "purpose", "verification_status", "loan_status"]:
    print(f"--- {c} ---")
    print(stg[c].value_counts(dropna=False).head(10).to_string(), "\n")

--- home_ownership ---
home_ownership
MORTGAGE    299890
RENT        273925
OWN          67027
OTHER           40
NONE            35
ANY              2 

--- purpose ---
purpose
debt_consolidation    365917
credit_card           159351
home_improvement       36689
other                  33517
major_purchase         12311
medical                 7036
small_business          6714
car                     6461
moving                  4491
vacation                4310 

--- verification_status ---
verification_status
Source Verified    231177
Not Verified       217316
Verified           192426 

--- loan_status ---
loan_status
Fully Paid            549933
Charged Off            90250
Current                  378
Late (31-120 days)       294
In Grace Period           35
Late (16-30 days)         29 



## Numeric ranges — any impossible values staging did not catch?

In [7]:
stg[["loan_amnt", "annual_inc", "dti", "revol_util", "fico_range_low", "fico_range_high"]].describe().round(1)

,loan_amnt,annual_inc,dti,revol_util,fico_range_low,fico_range_high
count,640919.0,640919.0,640916.0,640587.0,640919.0,640919.0
mean,12727.1,73478.4,17.9,53.8,694.2,698.2
std,7882.3,67863.5,8.5,23.7,30.6,30.6
min,1000.0,0.0,0.0,0.0,660.0,664.0
25%,7000.0,44000.0,11.6,36.3,670.0,674.0
50%,10000.0,61000.0,17.3,54.3,685.0,689.0
75%,16850.0,89000.0,23.7,72.0,710.0,714.0
max,35000.0,9000000.0,999.0,892.3,845.0,850.0


## Default rate by vintage — is the window benign but usable?

In [8]:
by_vintage = con.execute("""
    SELECT extract('year' FROM issue_d) AS issue_year,
           count(*) AS n,
           round(avg(CASE WHEN is_terminal AND loan_status <> 'Fully Paid' THEN 1.0
                          WHEN is_terminal THEN 0.0 END), 4) AS default_rate
    FROM stg_loans GROUP BY 1 ORDER BY 1
""").fetchdf()
by_vintage

,issue_year,n,default_rate
0,2012,43470,0.1358
1,2013,100422,0.1233
2,2014,162570,0.1373
3,2015,283173,0.1489
4,2016,51284,0.1484


**DQ decisions promoted to `sql/02_staging.sql`** (fill `reports/dq_log.md` from the
numbers above): 36-month filter, issue-year window, `min_months_since_issue`, impossible-
value guards on `loan_amnt` / `dti` / `revol_util`, de-dupe on `id`, drop
`Does not meet the credit policy%`. Median imputation for the nulls shown above happens
inside the model pipeline, not staging.

In [9]:
con.close()